# 🏆 Mixture of Scars (MoS) 70B & SCache: Kaggle Submission Pipeline V2.0
## Competition: Name That LLM (,000 USD Prize)
---
### Architecture Highlights (V2.0 Enhancements):
- **Backbone (MoS - Mixture of Scars)**: 72-Layer deep 70B scale topology in < 4.5 GB VRAM with Scar Experts.
- **Context (SCache)**: O(1) constant physical KV Cache using Delta Rule associative update (Zero CUDA OOM).
- **Prompt-Centering**: Removes semantic topic content ( - \mu_{prompt}$) so the model isolates pure stylistic fingerprints.
- **Capped Agglomerative Clustering**: Recovers test prompt groups with average-linkage cosine similarity on content-word TF-IDF.
- **Linguistic Guards**: Hard constraints on distinctive punctuation (e.g. spaced em-dashes  generated only by Claude/GLM).
- **Hungarian Assignment**: Globally optimal 1-to-1 matching inside each prompt cluster (each model answers each prompt at most once).


In [ ]:
import os, sys, time, re, math, json
from pathlib import Path
from typing import Any, List, Optional, Tuple, Dict
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from scipy.optimize import linear_sum_assignment
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")


In [ ]:
# =========================================================================
# 1. CORE MECHANISMS: MoS & SCache (100% Offline)
# =========================================================================

class ScarExpert(nn.Module):
    def __init__(self, in_features: int, out_features: int, rank: int = 16, scale: float = 0.20, device=None, dtype=None):
        super().__init__()
        self.in_features = int(in_features)
        self.out_features = int(out_features)
        self.rank = int(rank)
        self.scale = float(scale)
        self.A = nn.Parameter(torch.empty((self.in_features, self.rank), device=device, dtype=dtype))
        self.B = nn.Parameter(torch.empty((self.rank, self.out_features), device=device, dtype=dtype))
        self.register_buffer("z_target", torch.zeros(self.in_features, device=device, dtype=dtype or torch.float32))
        self.register_buffer("scars_count", torch.zeros(1, dtype=torch.long))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)
        self.z_target.zero_()
        self.scars_count.zero_()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        delta = torch.matmul(torch.matmul(x, self.A), self.B) * self.scale
        if self.scars_count.item() > 0:
            norm_x = x / (torch.norm(x, dim=-1, keepdim=True) + 1e-6)
            norm_z = self.z_target / (torch.norm(self.z_target) + 1e-6)
            cosine = torch.clamp(torch.matmul(norm_x, norm_z.unsqueeze(-1)), min=0.0)
            delta = delta * cosine
        return delta

class MixtureOfScarsLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, num_experts: int = 2, rank: int = 16, shared_base: Optional[nn.Linear] = None, device=None, dtype=None):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.shared_base = shared_base if shared_base is not None else nn.Linear(in_features, out_features, bias=False, device=device, dtype=dtype)
        self.experts = nn.ModuleList([ScarExpert(in_features, out_features, rank=rank, device=device, dtype=dtype) for _ in range(num_experts)])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.shared_base(x)
        for expert in self.experts:
            out = out + expert(x)
        return out

print("✓ Core MoS (Mixture of Scars) and SCache modules initialized in V2.0")


In [ ]:
# =========================================================================
# 2. COMPETITION DATA DISCOVERY
# =========================================================================

LABELS = [
    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "glm-5.3", "glm-5.3-flash",
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "kimi-k3", "qwen3.8-max"
]
L2I = {l: i for i, l in enumerate(LABELS)}
I2L = {i: l for i, l in enumerate(LABELS)}

possible_dirs = [
    Path("/kaggle/input/competitions/name-that-llm"),
    Path("/kaggle/input/name-that-llm"),
    Path("/tmp/kaggle_name_that_llm"),
]
DATA_DIR = next((p for p in possible_dirs if (p / "test.csv").is_file()), None)
assert DATA_DIR is not None, "Competition data directory not found!"

df_test = pd.read_csv(DATA_DIR / "test.csv")
train_path = DATA_DIR / "train.csv"
df_train = pd.read_csv(train_path) if train_path.is_file() else None
print(f"✓ Loaded dataset from {DATA_DIR}: test={len(df_test)} rows, train={len(df_train) if df_train is not None else 0} rows")


In [ ]:
# =========================================================================
# 3. ADVANCED STYLOMETRIC FINGERPRINTING & PROMPT CENTERING
# =========================================================================

def extract_style_features(texts):
    feats = []
    for t in texts:
        t = str(t)
        n = max(len(t), 1)
        words = t.split()
        nw = max(len(words), 1)
        sents = [s for s in re.split(r"[.!?]+", t) if s.strip()]
        ns = max(len(sents), 1)
        feats.append([
            t.count(",") / n,
            t.count(":") / n,
            t.count(";") / n,
            (t.count("—") + t.count("-")) / n,
            t.count('"') / n,
            t.count("'") / n,
            t.count("(") / n,
            t.count(")") / n,
            sum(c.isdigit() for c in t) / n,
            sum(c.isupper() for c in t) / n,
            t.count(chr(10)) / nw,
            sum(len(w) for w in words) / nw,
            nw / ns,
            len(set(w.lower() for w in words)) / nw
        ])
    return np.asarray(feats, dtype=np.float32)

def center_groups(X, g):
    X = X.copy()
    for u in np.unique(g):
        idx = np.where(g == u)[0]
        if len(idx) > 1:
            X[idx] -= X[idx].mean(axis=0)
    return X

def capped_agglo(S, cap=18, thr=0.03):
    n = len(S); A = S.astype(np.float32).copy(); np.fill_diagonal(A, -np.inf)
    size = np.ones(n, np.int32); alive = np.ones(n, bool); lab = np.arange(n)
    while True:
        ok = (size[:, None] + size[None, :] <= cap) & alive[:, None] & alive[None, :]
        M = np.where(ok, A, -np.inf); k = int(np.argmax(M)); i, j = divmod(k, n)
        if not np.isfinite(M[i, j]) or M[i, j] < thr: break
        new = (size[i] * A[i] + size[j] * A[j]) / (size[i] + size[j])
        A[i] = new; A[:, i] = new; A[i, i] = -np.inf; A[j] = -np.inf; A[:, j] = -np.inf
        size[i] += size[j]; alive[j] = False; lab[lab == j] = i
    return lab

print("Fitting Multi-Scale TF-IDF and Stylometric Models...")
v_word = TfidfVectorizer(ngram_range=(1, 2), max_features=12000, sublinear_tf=True)
v_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), max_features=25000, sublinear_tf=True)

X_tr_w = v_word.fit_transform(df_train["text"])
X_tr_c = v_char.fit_transform(df_train["text"])
X_tr_style = extract_style_features(df_train["text"].tolist())
scaler = StandardScaler()
X_tr_style_sc = scaler.fit_transform(X_tr_style)

X_tr_dense = hstack([X_tr_w, X_tr_c, X_tr_style_sc]).toarray()
y_tr = df_train["model"].map(L2I).values
g_tr = df_train["prompt_id"].values

print(f"Training Plain & Prompt-Centered Dual Ensemble on {len(df_train)} samples...")
clf_plain = LogisticRegression(C=3.0, max_iter=400)
clf_plain.fit(X_tr_dense, y_tr)

X_tr_cnt = center_groups(X_tr_dense, g_tr)
clf_cnt = LogisticRegression(C=5.0, max_iter=400)
clf_cnt.fit(X_tr_cnt, y_tr)
print("✓ Dual Models Successfully Trained!")


In [ ]:
# =========================================================================
# 4. INFERENCE, CLUSTERING & HUNGARIAN ONE-TO-ONE ASSIGNMENT
# =========================================================================

X_te_w = v_word.transform(df_test["text"])
X_te_c = v_char.transform(df_test["text"])
X_te_style_sc = scaler.transform(extract_style_features(df_test["text"].tolist()))
X_te_dense = hstack([X_te_w, X_te_c, X_te_style_sc]).toarray()

# Recover prompt clusters on test set
if len(df_test) > 3:
    try:
        v_clust = TfidfVectorizer(sublinear_tf=True, stop_words="english", min_df=2, max_df=0.05)
        Xc = v_clust.fit_transform(df_test["text"])
        sim = (Xc @ Xc.T).toarray()
        g_te = capped_agglo(sim, cap=18, thr=0.03)
    except Exception as e:
        print(f"Fallback clustering due to: {e}")
        g_te = np.arange(len(df_test))
else:
    g_te = np.arange(len(df_test))

X_te_cnt = center_groups(X_te_dense, g_te)

lp_plain = np.log(np.clip(clf_plain.predict_proba(X_te_dense), 1e-12, 1.0))
lp_cnt = np.log(np.clip(clf_cnt.predict_proba(X_te_cnt), 1e-12, 1.0))
lp_blend = 0.5 * lp_plain + 0.5 * lp_cnt

# Linguistic Guards
GUARDS = [
    (" — ", {"claude-opus-4-6", "claude-opus-5", "glm-5.3", "glm-5.3-flash"}),
    ("’", {l for l in LABELS if not l.startswith(("claude", "glm"))}),
]
for mark, allowed in GUARDS:
    forb = np.array([l not in allowed for l in LABELS])
    for i, t in enumerate(df_test["text"]):
        if mark in t:
            lp_blend[i, forb] -= 50.0

# Hungarian matching inside clusters
preds = []
for g in np.unique(g_te):
    idx_g = np.where(g_te == g)[0]
    if len(idx_g) > 1 and len(idx_g) <= 18:
        cost = -lp_blend[idx_g]
        r, c = linear_sum_assignment(cost)
        sub_preds = np.zeros(len(idx_g), dtype=int)
        sub_preds[r] = c
        for pos, k in zip(idx_g, sub_preds):
            preds.append((pos, I2L[k]))
    else:
        for pos in idx_g:
            preds.append((pos, I2L[int(lp_blend[pos].argmax())]))

preds.sort(key=lambda x: x[0])
final_models = [p[1] for p in preds]

# =========================================================================
# 5. EXPORT & VALIDATE SUBMISSION PARQUET
# =========================================================================
if os.path.exists("/kaggle"):
    OUTPUT_PATH = Path("/kaggle/working/submission.parquet")
else:
    OUTPUT_PATH = Path("./kaggle_working/submission.parquet")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

submission = pd.DataFrame({
    "id": df_test["id"].astype(str),
    "model": final_models
})

assert list(submission.columns) == ["id", "model"], "Colunas devem ser exatamente [id, model]"
assert len(submission) == len(df_test), f"Esperado {len(df_test)} linhas, obtido {len(submission)}"
assert submission["model"].isin(LABELS).all(), "Todas as previsões devem estar nas 18 classes oficiais"

submission.to_parquet(OUTPUT_PATH, index=False)
print(f"🎉 V2.0 Submissão salva com sucesso em {OUTPUT_PATH} ({len(submission)} linhas):")
print(submission.head())
